# =========================================
# project_root/
#   data/
#       la_crime_with_acs.csv   # your input
#   outputs/
#   src/
#       __init__.py
#       config.py
#       preprocess.py
#       features.py
#       clustering.py
#       modeling.py
#       explain.py
#   main.py
# =========================================

In [26]:
# =========================================
# File: src/config.py
# =========================================

from __future__ import annotations
from dataclasses import dataclass
from typing import Optional


@dataclass
class Config:
    # Paths
    csv_path: str = "data/crime_with_acs.csv"
    out_dir: str = "outputs"

    # Raw columns
    date_col: str = "DATE OCC"
    time_col: str = "TIME OCC"
    lat_col: str = "lat"
    lon_col: str = "lon"
    violent_col: str = "Crm Cd Desc"  # description-based violent flag
    is_violent_col: Optional[str] = None  # if you later create a boolean violent column

    # Violent crime regex on description (fallback)
    ''' violent_regex: str = (
        r"(HOMICIDE|MURDER|ROBBERY|ASSAULT|AGGRAVATED|RAPE|SEXUAL ASSAULT|SHOOTING|"  # noqa: E501
        r"BATTERY W/ SERIOUS|ADW|SHOTS FIRED)"
    )'''

    violent_regex: str = (
    r"(HOMICIDE|MURDER|MANSLAUGHTER|"  # 殺人
    r"ROBBERY|ARMED ROBBERY|"           # 搶劫
    r"ASSAULT|AGGRAVATED|BATTERY|ADW|"  # 攻擊 (包含所有 battery)
    r"RAPE|SEXUAL|SODOMY|PENETRATION|"  # 性犯罪
    r"SHOOTING|SHOTS FIRED|DISCHARGE|"  # 槍擊
    r"KIDNAP|CHILD ABUSE|"              # 綁架/虐童
    r"WEAPON|DEADLY)"                   # 武器相關
)

    # Time filter
    since: str = "2024-01-01"  # use only incidents since 2020

    # Grid settings
    grid: str = "h3"  # "h3" or "bin"
    h3_res: int = 9
    bin_deg: float = 0.005  # ~500m at LA latitude if using bins

    # Clustering
    kmeans_k_min: int = 3
    kmeans_k_max: int = 10
    dbscan_eps: float = 0.0025  # ~250m in degrees
    dbscan_min_samples: int = 15

    # Labeling high-risk grid-cell-hours
    label_rule: str = "count>=1"  # "quantile" or "count>=1"
    risk_quantile: float = 0.95

    # Train/test split
    test_size_days: int = 30

    # Modeling
    use_xgboost: bool = True


In [27]:
# =========================================
# File: src/preprocess.py
# =========================================

from __future__ import annotations
import os
import re
from typing import Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

from src.config import Config

# ✅ Auto-detect H3 version (works for both v3.x and v4.x+)
H3_OK = False
_geo_to_h3 = None

try:
    # New API (h3>=4)
    from h3 import latlng_to_cell as geo_to_h3
    _geo_to_h3 = geo_to_h3
    H3_OK = True
except Exception:
    try:
        # Mid API (h3.api.basic_int)
        from h3.api import basic_int
        if hasattr(basic_int, "geo_to_h3"):
            _geo_to_h3 = basic_int.geo_to_h3
            H3_OK = True
    except Exception:
        try:
            # Old API (h3==3.7.6 or similar)
            import h3
            if hasattr(h3, "geo_to_h3"):
                _geo_to_h3 = h3.geo_to_h3
                H3_OK = True
        except Exception:
            H3_OK = False



def parse_time_hhmm_str(val) -> Tuple[int, int]:
    """Parse LAPD TIME OCC (e.g., '140', '0', '2359') into (hour, minute)."""
    if pd.isna(val):
        return 0, 0
    s = str(val).strip()
    if not s.isdigit():
        return 0, 0
    if len(s) <= 2:
        h, m = 0, int(s)
    elif len(s) == 3:
        h, m = int(s[0]), int(s[1:])
    else:
        h, m = int(s[:-2]), int(s[-2:])
    h = max(0, min(23, h))
    m = max(0, min(59, m))
    return h, m


def to_datetime(row, cfg: Config) -> pd.Timestamp:
    d = pd.to_datetime(row[cfg.date_col], errors="coerce")
    hh, mm = parse_time_hhmm_str(row[cfg.time_col])
    if pd.isna(d):
        return pd.NaT
    return pd.Timestamp(d.year, d.month, d.day, hh, mm)


def flag_violent(df: pd.DataFrame, cfg: Config) -> pd.Series:
    if cfg.is_violent_col and cfg.is_violent_col in df.columns:
        return df[cfg.is_violent_col].astype(bool)
    if cfg.violent_col and cfg.violent_col in df.columns:
        patt = re.compile(cfg.violent_regex, flags=re.I)
        return df[cfg.violent_col].fillna("").astype(str).str.contains(patt)
    return pd.Series(True, index=df.index)


def build_grid_id(df: pd.DataFrame, cfg: Config) -> pd.Series:
    """Convert lat/lon to grid id using h3 (if available) else lat/lon binning."""
    if H3_OK and _geo_to_h3 is not None:
        try:
            return df.apply(
                lambda r: _geo_to_h3(float(r[cfg.lat_col]), float(r[cfg.lon_col]), cfg.h3_res),
                axis=1,
            )
        except Exception as e:
            print(f"⚠️ H3 failed, fallback to lat/lon bins: {e}")
    # fallback: approximate grid bins
    lat_bin = np.floor(df[cfg.lat_col] / cfg.bin_deg).astype(int)
    lon_bin = np.floor(df[cfg.lon_col] / cfg.bin_deg).astype(int)
    return lat_bin.astype(str) + ":" + lon_bin.astype(str)



def load_and_preprocess(cfg: Config) -> pd.DataFrame:
    if not os.path.exists(cfg.csv_path):
        raise FileNotFoundError(f"CSV not found: {cfg.csv_path}")

    df = pd.read_csv(cfg.csv_path)

    # Ensure needed columns exist
    needed = [cfg.date_col, cfg.time_col, cfg.lat_col, cfg.lon_col]
    for c in [cfg.violent_col, cfg.is_violent_col]:
        if c and c not in needed:
            needed.append(c)
    df = df[[c for c in needed if c in df.columns]].copy()

    # Parse datetime
    tqdm.pandas(desc="Parsing timestamps")
    df["ts"] = df.progress_apply(lambda r: to_datetime(r, cfg), axis=1)
    df = df.dropna(subset=["ts"])

    # Filter since
    if cfg.since:
        df = df[df["ts"] >= pd.to_datetime(cfg.since)].copy()

    # Clean lat/lon
    df[cfg.lat_col] = pd.to_numeric(df[cfg.lat_col], errors="coerce")
    df[cfg.lon_col] = pd.to_numeric(df[cfg.lon_col], errors="coerce")
    df = df.dropna(subset=[cfg.lat_col, cfg.lon_col])

    # Violent flag
    df["is_violent"] = flag_violent(df, cfg)

    # Grid
    df["grid_id"] = build_grid_id(df, cfg)

    return df

In [5]:
# =========================================
# File: src/features.py
# =========================================

from __future__ import annotations
import pandas as pd
import numpy as np

from src.config import Config


def aggregate_to_grid_hour(violent_df: pd.DataFrame) -> pd.DataFrame:
    violent_df = violent_df.dropna(subset=["ts"]).copy()
    violent_df["hour_ts"] = violent_df["ts"].dt.floor("H")
    agg = (
        violent_df.groupby(["grid_id", "hour_ts"], as_index=False)
        .size()
        .rename(columns={"size": "cnt"})
    )
    return agg


def make_panel_and_label(agg: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    # Full grid-hour panel
    tmin, tmax = agg["hour_ts"].min(), agg["hour_ts"].max()
    grids = agg["grid_id"].unique()
    full_index = pd.MultiIndex.from_product(
        [grids, pd.date_range(tmin, tmax, freq="H")], names=["grid_id", "hour_ts"]
    )
    panel = (
        agg.set_index(["grid_id", "hour_ts"])
        .reindex(full_index)
        .reset_index()
    )
    panel["cnt"] = panel["cnt"].fillna(0)

    # Labels
    if cfg.label_rule == "quantile":
        thr = panel.groupby("hour_ts")["cnt"].quantile(cfg.risk_quantile).rename("thr")
        panel = panel.merge(thr, on="hour_ts", how="left")
        panel["y"] = (panel["cnt"] >= panel["thr"].fillna(1)).astype(int)
        panel = panel.drop(columns=["thr"])
    else:
        panel["y"] = (panel["cnt"] >= 1).astype(int)

    # Sort for rolling features
    panel = panel.sort_values(["grid_id", "hour_ts"]).reset_index(drop=True)

    def add_lags(gdf: pd.DataFrame) -> pd.DataFrame:
        gdf["lag_1h"] = gdf["cnt"].shift(1)
        gdf["lag_6h"] = gdf["cnt"].rolling(6).sum().shift(1)
        gdf["lag_24h"] = gdf["cnt"].rolling(24).sum().shift(1)
        gdf["lag_168h"] = gdf["cnt"].rolling(168).sum().shift(1)
        gdf["ma_24h"] = gdf["cnt"].rolling(24).mean().shift(1)
        return gdf

    panel = panel.groupby("grid_id", group_keys=False).apply(add_lags)

    # Calendar features
    panel["hour"] = panel["hour_ts"].dt.hour
    panel["dow"] = panel["hour_ts"].dt.dayofweek
    panel["month"] = panel["hour_ts"].dt.month
    panel["is_weekend"] = (panel["dow"] >= 5).astype(int)

    # Historical mean
    hist = (
        panel.groupby("grid_id")["cnt"]
        .expanding()
        .mean()
        .reset_index(level=0, drop=True)
        .shift(1)
    )
    panel["hist_mean"] = hist.values

    for c in ["lag_1h", "lag_6h", "lag_24h", "lag_168h", "ma_24h", "hist_mean"]:
        panel[c] = panel[c].fillna(0)

    return panel


def temporal_train_test_split(panel: pd.DataFrame, cfg: Config):
    """
    Split panel into train/test based on time.
    """
    # 🔧 修正：確保 hour_ts 是 datetime 格式
    if panel["hour_ts"].dtype == 'object':  # 如果是字串
        panel["hour_ts"] = pd.to_datetime(panel["hour_ts"])
    
    cutoff = panel["hour_ts"].max() - pd.Timedelta(days=cfg.test_size_days)
    
    train = panel[panel["hour_ts"] < cutoff].copy()
    test = panel[panel["hour_ts"] >= cutoff].copy()
    
    return train, test

In [29]:
# =========================================
# File: src/clustering.py
# =========================================

from __future__ import annotations
import os

import numpy as np
import pandas as pd

from src.config import Config

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_OK = True
except Exception:
    PLOTLY_OK = False

try:
    import folium  # type: ignore
    FOLIUM_OK = True
except Exception:
    FOLIUM_OK = False


def descriptive_analysis(violent_df: pd.DataFrame, cfg: Config) -> None:
    os.makedirs(cfg.out_dir, exist_ok=True)

    violent_df["hour"] = violent_df["ts"].dt.hour
    violent_df["dow"] = violent_df["ts"].dt.dayofweek

    heat = (
        violent_df.groupby(["dow", "hour"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )

    if PLOTLY_OK:
        fig = px.density_heatmap(
            heat,
            x="hour",
            y="dow",
            z="count",
            color_continuous_scale="Inferno",
            category_orders={"dow": [0, 1, 2, 3, 4, 5, 6]},
            title="Violent incidents: Hour x Day-of-Week",
        )
        fig.update_yaxes(
            tickvals=list(range(7)),
            ticktext=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
        )
        fig.write_html(os.path.join(cfg.out_dir, "heatmap_hour_dow.html"))

    monthly = (
        violent_df.set_index("ts")
        .resample("MS")
        .size()
        .rename("count")
        .reset_index()
    )

    if PLOTLY_OK:
        fig2 = px.line(monthly, x="ts", y="count", title="Monthly violent incidents (since 2020)")
        fig2.write_html(os.path.join(cfg.out_dir, "monthly_trend.html"))


def clustering_hotspots(violent_df: pd.DataFrame, cfg: Config) -> None:
    os.makedirs(cfg.out_dir, exist_ok=True)

    recent_cut = violent_df["ts"].max() - pd.Timedelta(days=90)
    df_recent = violent_df[violent_df["ts"] >= recent_cut].copy()

    df_recent["hour"] = df_recent["ts"].dt.hour
    X = df_recent[[cfg.lat_col, cfg.lon_col, "hour"]].to_numpy()
    X_std = StandardScaler().fit_transform(X)

    Ks = list(range(cfg.kmeans_k_min, cfg.kmeans_k_max + 1))
    inertias, sils = [], []

    for k in Ks:
        km = KMeans(n_clusters=k, n_init=10, random_state=42)
        labels = km.fit_predict(X_std)
        inertias.append(km.inertia_)

        if len(X_std) > 5000:
            idx = np.random.choice(len(X_std), 5000, replace=False)
            sil = silhouette_score(X_std[idx], labels[idx])
        else:
            sil = silhouette_score(X_std, labels)
        sils.append(sil)

    if PLOTLY_OK:
        fig_elbow = go.Figure(data=[go.Scatter(x=Ks, y=inertias, mode="lines+markers")])
        fig_elbow.update_layout(
            title="KMeans Elbow (Inertia)",
            xaxis_title="K",
            yaxis_title="Inertia",
        )
        fig_elbow.write_html(os.path.join(cfg.out_dir, "kmeans_elbow.html"))

        fig_sil = go.Figure(data=[go.Scatter(x=Ks, y=sils, mode="lines+markers")])
        fig_sil.update_layout(
            title="KMeans Silhouette",
            xaxis_title="K",
            yaxis_title="Silhouette Score",
        )
        fig_sil.write_html(os.path.join(cfg.out_dir, "kmeans_silhouette.html"))

    # DBSCAN on lat/lon
    X_ll = df_recent[[cfg.lat_col, cfg.lon_col]].to_numpy()
    db = DBSCAN(
        eps=cfg.dbscan_eps,
        min_samples=cfg.dbscan_min_samples,
        n_jobs=-1
    )

    db_labels = db.fit_predict(X_ll)
    df_recent["db_label"] = db_labels

    if FOLIUM_OK:
        center = [df_recent[cfg.lat_col].mean(), df_recent[cfg.lon_col].mean()]
        m = folium.Map(location=center, zoom_start=12)
        for _, r in df_recent.iterrows():
            color = "red" if r["db_label"] >= 0 else "gray"
            folium.CircleMarker(
                location=[r[cfg.lat_col], r[cfg.lon_col]],
                radius=2,
                color=color,
                fill=True,
                fill_opacity=0.6,
            ).add_to(m)
        m.save(os.path.join(cfg.out_dir, "dbscan_clusters_map.html"))

In [30]:
# =========================================
# File: src/modeling.py
# =========================================

from __future__ import annotations
from typing import Dict, Any

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
)

try:
    from xgboost import XGBClassifier  # type: ignore
    XGB_OK = True
except Exception:
    XGB_OK = False

from src.config import Config


FEATURE_COLS = [
    "lag_1h",
    "lag_6h",
    "lag_24h",
    "lag_168h",
    "ma_24h",
    "hist_mean",
    "hour",
    "dow",
    "month",
    "is_weekend",
]


def train_models(train: pd.DataFrame, test: pd.DataFrame, cfg: Config) -> Dict[str, Any]:
    X_tr = train[FEATURE_COLS].values
    y_tr = train["y"].values
    X_te = test[FEATURE_COLS].values
    y_te = test["y"].values

    results: Dict[str, Any] = {}

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, class_weight="balanced")
    lr.fit(X_tr, y_tr)
    p_lr = lr.predict_proba(X_te)[:, 1]
    results["logreg"] = {
        "model": lr,
        "roc_auc": roc_auc_score(y_te, p_lr),
        "pr_auc": average_precision_score(y_te, p_lr),
        "f1@0.5": f1_score(y_te, (p_lr >= 0.5).astype(int)),
        "proba": p_lr,
    }

    # Random Forest
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        n_jobs=-1,
        class_weight="balanced_subsample",
        random_state=42,
    )
    rf.fit(X_tr, y_tr)
    p_rf = rf.predict_proba(X_te)[:, 1]
    results["rf"] = {
        "model": rf,
        "roc_auc": roc_auc_score(y_te, p_rf),
        "pr_auc": average_precision_score(y_te, p_rf),
        "f1@0.5": f1_score(y_te, (p_rf >= 0.5).astype(int)),
        "proba": p_rf,
    }

    # XGBoost (optional)
    if cfg.use_xgboost and XGB_OK:
        xgb = XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42,
            tree_method="hist",
        )
        xgb.fit(X_tr, y_tr)
        p_xgb = xgb.predict_proba(X_te)[:, 1]
        results["xgb"] = {
            "model": xgb,
            "roc_auc": roc_auc_score(y_te, p_xgb),
            "pr_auc": average_precision_score(y_te, p_xgb),
            "f1@0.5": f1_score(y_te, (p_xgb >= 0.5).astype(int)),
            "proba": p_xgb,
        }

    # Recall@K using best PR-AUC model
    for k in [100, 500, 1000]:
        best_name = max(results, key=lambda m: results[m]["pr_auc"])
        proba = results[best_name]["proba"]
        top_idx = np.argpartition(proba, -k)[-k:]
        recall_at_k = test.iloc[top_idx]["y"].mean()
        results[best_name][f"recall@{k}"] = float(recall_at_k)

    return results

In [31]:
# =========================================
# File: src/explain.py
# =========================================

from __future__ import annotations
import os

import pandas as pd

from src.config import Config
from src.modeling import FEATURE_COLS

try:
    import shap  # type: ignore
    SHAP_OK = True
except Exception:
    SHAP_OK = False


def shap_explain(best_model, train: pd.DataFrame, cfg: Config) -> None:
    if not SHAP_OK:
        return

    X = train[FEATURE_COLS]

    try:
        explainer = shap.Explainer(best_model, X)
        shap_values = explainer(X[:2000])
        shap.plots.beeswarm(shap_values, show=False)
        import matplotlib.pyplot as plt

        plt.tight_layout()
        os.makedirs(cfg.out_dir, exist_ok=True)
        plt.savefig(os.path.join(cfg.out_dir, "shap_beeswarm.png"), dpi=160)
        plt.close()
    except Exception:
        # Fail silently if SHAP plotting fails
        pass

In [32]:
# ===== Cell 1: 環境設置與導入 =====
from __future__ import annotations
import sys
import subprocess
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# Auto-install missing modules
REQUIRED = ["pandas", "numpy", "tqdm", "scikit-learn"]
OPTIONAL = ["h3", "plotly", "folium", "xgboost", "shap"]

print("🔍 Checking packages...")
for pkg in REQUIRED:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

for pkg in OPTIONAL:
    try:
        __import__(pkg)
    except ImportError:
        print(f"⚠️  Optional: {pkg} not found")

# 導入專案模組
if 'src' not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

from src.config import Config
from src.preprocess import load_and_preprocess
from src.features import (
    aggregate_to_grid_hour,
    make_panel_and_label,
    temporal_train_test_split,
)
from src.clustering import descriptive_analysis, clustering_hotspots
from src.modeling import train_models
from src.explain import shap_explain

# 初始化設定
cfg = Config()
os.makedirs(cfg.out_dir, exist_ok=True)

print(f"✅ Setup complete! Output dir: {cfg.out_dir}")


🔍 Checking packages...
Installing scikit-learn...
✅ Setup complete! Output dir: outputs



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [37]:
# ===== Cell 2: Step 1 - Load + Preprocess =====
print("=" * 60)
print("STEP 1: Load and Preprocess Data")
print("=" * 60)

df = load_and_preprocess(cfg)
#violent_df = df[df["is_violent"]].copy()
violent_df = df.copy()

print(f"\n✅ Data loaded:")
print(f"   Total crimes: {len(df):,}")
print(f"   Violent crimes: {len(violent_df):,} ({len(violent_df)/len(df)*100:.2f}%)")
print(f"\n📊 Sample data:")
display(df.head())
print(f"\n📊 Violent crime sample:")
display(violent_df.head())

STEP 1: Load and Preprocess Data


Parsing timestamps: 100%|██████████| 924869/924869 [02:23<00:00, 6461.74it/s]
/Users/randolphyu/Documents/DSCI550/Project/la_crime_report_pipeline/src/preprocess.py:75: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return df[cfg.violent_col].fillna("").astype(str).str.contains(patt)



✅ Data loaded:
   Total crimes: 116,127
   Violent crimes: 116,127 (100.00%)

📊 Sample data:


,DATE OCC,TIME OCC,lat,lon,Crm Cd Desc,ts,is_violent,grid_id
11,08/16/2024 12:00:00 AM,1500,34.039559,-118.213976,VEHICLE - STOLEN,2024-08-16 15:00:00,False,8929a1d747bffff
20,04/11/2024 12:00:00 AM,1030,34.156903,-118.605813,DISTURBING THE PEACE,2024-04-11 10:30:00,False,8929a18191bffff
28,01/13/2024 12:00:00 AM,1,34.042193,-118.215429,THEFT PLAIN - PETTY ($950 & UNDER),2024-01-13 00:01:00,False,8929a1d7463ffff
40,09/19/2024 12:00:00 AM,1900,34.067056,-118.280625,VEHICLE - STOLEN,2024-09-19 19:00:00,False,8929a1d635bffff
50,09/26/2024 12:00:00 AM,101,34.268537,-118.447515,THEFT OF IDENTITY,2024-09-26 01:01:00,False,8929a112617ffff



📊 Violent crime sample:


,DATE OCC,TIME OCC,lat,lon,Crm Cd Desc,ts,is_violent,grid_id
11,08/16/2024 12:00:00 AM,1500,34.039559,-118.213976,VEHICLE - STOLEN,2024-08-16 15:00:00,False,8929a1d747bffff
20,04/11/2024 12:00:00 AM,1030,34.156903,-118.605813,DISTURBING THE PEACE,2024-04-11 10:30:00,False,8929a18191bffff
28,01/13/2024 12:00:00 AM,1,34.042193,-118.215429,THEFT PLAIN - PETTY ($950 & UNDER),2024-01-13 00:01:00,False,8929a1d7463ffff
40,09/19/2024 12:00:00 AM,1900,34.067056,-118.280625,VEHICLE - STOLEN,2024-09-19 19:00:00,False,8929a1d635bffff
50,09/26/2024 12:00:00 AM,101,34.268537,-118.447515,THEFT OF IDENTITY,2024-09-26 01:01:00,False,8929a112617ffff


# ===== Cell 3: Step 2 - Descriptive Analysis (加強版) =====
print("=" * 60)
print("STEP 2: Descriptive Analysis")
print("=" * 60)

# 先看一下 violent_df 的基本統計
print(f"\n📊 Violent Crime Overview:")
print(f"   Total records: {len(violent_df):,}")
print(f"   Date range: {violent_df['ts'].min()} to {violent_df['ts'].max()}")
print(f"   Unique grids: {violent_df['grid_id'].nunique():,}")

# 如果有 violent_col，顯示犯罪類型分佈
if cfg.violent_col and cfg.violent_col in violent_df.columns:
    print(f"\n🔴 Top 10 Violent Crime Types:")
    print(violent_df[cfg.violent_col].value_counts().head(10))

# 時間分佈
print(f"\n📅 Temporal Distribution:")
violent_df['year'] = violent_df['ts'].dt.year
violent_df['month'] = violent_df['ts'].dt.month
violent_df['hour'] = violent_df['ts'].dt.hour
violent_df['day_of_week'] = violent_df['ts'].dt.dayofweek

print("\nBy Year:")
print(violent_df['year'].value_counts().sort_index())

print("\nBy Hour of Day:")
print(violent_df['hour'].value_counts().sort_index())

print("\nBy Day of Week (0=Mon, 6=Sun):")
print(violent_df['day_of_week'].value_counts().sort_index())

# 空間分佈
print(f"\n🗺️  Spatial Distribution:")
print(f"   Top 10 hotspot grids:")
print(violent_df['grid_id'].value_counts().head(10))

# 執行原本的函數
print(f"\n⏱️  Running descriptive_analysis()...")
descriptive_analysis(violent_df, cfg)

print(f"\n✅ Descriptive analysis complete!")
print(f"   Output directory: {cfg.out_dir}")

# 列出生成的檔案
import os
if os.path.exists(cfg.out_dir):
    files = os.listdir(cfg.out_dir)
    if files:
        print(f"   Generated files:")
        for f in files:
            print(f"      - {f}")
    else:
        print(f"   ⚠️  No files found in output directory")

In [8]:
# ===== 診斷腳本 =====
import os
import pandas as pd
from src.config import Config

cfg = Config()

# 1. 檢查已生成的檔案
print("📁 Checking generated files:")
files_expected = [
    "grid_hour_counts.csv",
    "panel_labeled.csv",
    "metrics.csv"
]

for fname in files_expected:
    fpath = os.path.join(cfg.out_dir, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f"   ✅ {fname} ({size:.1f} KB)")
    else:
        print(f"   ❌ {fname} NOT FOUND")

# 2. 檢查 panel_labeled.csv
panel_path = os.path.join(cfg.out_dir, "panel_labeled.csv")
if os.path.exists(panel_path):
    print(f"\n📊 Loading panel data...")
    panel = pd.read_csv(panel_path)
    print(f"   Shape: {panel.shape}")
    print(f"   Columns: {list(panel.columns)}")
    print(f"   Memory: {panel.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    if 'y' in panel.columns:
        print(f"\n   Label distribution:")
        print(panel['y'].value_counts())
        print(f"   Positive rate: {panel['y'].mean()*100:.2f}%")
    else:
        print("   ⚠️ WARNING: No 'y' column found!")
    
    # 嘗試執行 Step 5
    print(f"\n🧪 Testing Step 5...")
    try:
        from src.features import temporal_train_test_split
        train, test = temporal_train_test_split(panel, cfg)
        print(f"   ✅ Train/test split successful")
        print(f"      Train: {len(train):,}, Test: {len(test):,}")
        
        # 測試 modeling
        print(f"\n🧪 Testing modeling...")
        from src.modeling import train_models
        results = train_models(train, test, cfg)
        print(f"   ✅ Modeling successful!")
        print(f"   Models: {list(results.keys())}")
        
    except Exception as e:
        print(f"   ❌ ERROR: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"\n❌ panel_labeled.csv not found!")

📁 Checking generated files:
   ✅ grid_hour_counts.csv (4091.1 KB)
   ✅ panel_labeled.csv (8780774.3 KB)
   ❌ metrics.csv NOT FOUND

📊 Loading panel data...


KeyboardInterrupt: 

In [7]:
# ===== 在執行 temporal_train_test_split 之前 =====
import pandas as pd
from src.config import Config

cfg = Config()

# 1. 載入 panel
panel = pd.read_csv("outputs/panel_labeled.csv")

print("🔍 Before conversion:")
print(f"   hour_ts dtype: {panel['hour_ts'].dtype}")
print(f"   Sample value: {panel['hour_ts'].iloc[0]}")
print(f"   Sample type: {type(panel['hour_ts'].iloc[0])}")

# 2. 轉換成 datetime
panel["hour_ts"] = pd.to_datetime(panel["hour_ts"], errors='coerce')

print("\n✅ After conversion:")
print(f"   hour_ts dtype: {panel['hour_ts'].dtype}")
print(f"   Sample value: {panel['hour_ts'].iloc[0]}")
print(f"   Sample type: {type(panel['hour_ts'].iloc[0])}")

# 3. 檢查是否有 NaT (Not a Time)
if panel["hour_ts"].isna().any():
    print(f"\n⚠️  Warning: {panel['hour_ts'].isna().sum()} invalid timestamps")
    panel = panel.dropna(subset=["hour_ts"])

# 4. 現在測試 split
print("\n🧪 Testing temporal split...")
from src.features import temporal_train_test_split

try:
    train, test = temporal_train_test_split(panel, cfg)
    print(f"✅ SUCCESS!")
    print(f"   Train: {len(train):,}")
    print(f"   Test: {len(test):,}")
except Exception as e:
    print(f"❌ Still failed: {e}")
    import traceback
    traceback.print_exc()

🔍 Before conversion:
   hour_ts dtype: object
   Sample value: 2024-01-01 00:00:00
   Sample type: <class 'str'>

✅ After conversion:
   hour_ts dtype: datetime64[ns]
   Sample value: 2024-01-01 00:00:00
   Sample type: <class 'pandas._libs.tslibs.timestamps.Timestamp'>

🧪 Testing temporal split...
✅ SUCCESS!
   Train: 92,724,648
   Test: 5,751,417
